# Income model: feature analysis

This notebook explains **why model 2 uses only `age`** to predict income.
It is for analysis only. The pipeline that runs on the VM is `src/main.py`.

Run it from top to bottom (**Restart & Run All**). Each section builds on the previous one.

| Section | Question |
|---|---|
| 0 | Setup |
| 1 | Load the data once |
| 2 | What does the target look like? What is the baseline? |
| 3 | Which single feature is related to income? |
| 4 | Does combining age with gender help? |
| 5 | What is the best accuracy possible with age? |
| 6 | Which feature set and model work best on the test set? |
| 7 | Check the final code in `src/income_model.py` |
| 8 | Numbers for the report |

## 0. Setup

In [13]:
import os
import sys
from pathlib import Path
from getpass import getpass

# Make `from db import ...` work: add src/ to the module search path.
# The notebook may start in notebooks/ or in the project root.
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

# db.py reads DB_PASSWORD when it is imported, so set it first.
if not os.environ.get("DB_PASSWORD"):
    os.environ["DB_PASSWORD"] = getpass("DB password: ")

%load_ext autoreload
%autoreload 2

import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

SEEDS = list(range(10))          # seeds used for every comparison
TARGET_ACC = 0.35                # goal from the assignment

# Readable order for the interval labels (string sort would put "100k" before "10k").
INCOME_ORDER = ["0-10k", "10k-20k", "20k-40k", "40k-60k",
                "60k-100k", "100k-150k", "150k-250k", "250k+"]
AGE_ORDER = ["0-16", "17-25", "26-35", "36-45", "46-55", "56-65", "66-75", "76+"]

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Load the data

Load once. Every later cell uses `df` from memory, so there is no need to reconnect.

In [14]:
from db import load_requests

df = load_requests()
df["hour"] = pd.to_datetime(df["time_of_day"]).dt.hour

print(f"{len(df)} rows")
df.isna().sum()

99999 rows


id                0
client_ip         0
country           0
is_banned         0
gender            0
age               0
income            0
time_of_day       0
requested_file    0
hour              0
dtype: int64

## 2. Target: income

The most common class gives the **baseline**: the accuracy of always guessing that class.
A useful model must beat it.

In [15]:
income_dist = df["income"].value_counts(normalize=True).reindex(INCOME_ORDER)
baseline = income_dist.max()

print(f"{df['income'].nunique()} income levels, baseline = {baseline:.4f}")
income_dist.round(4)

8 income levels, baseline = 0.1274


income
0-10k        0.1274
10k-20k      0.1249
20k-40k      0.1243
40k-60k      0.1252
60k-100k     0.1243
100k-150k    0.1254
150k-250k    0.1233
250k+        0.1252
Name: proportion, dtype: float64

## 3. One feature at a time

For each feature, compare the income distribution inside each group with the overall distribution.

- **max_deviation** = the largest gap between a group's share and the overall share.
  Close to 0 means the feature tells us nothing about income.
- For features with many values (`country`, `requested_file`), groups are small, so the
  deviation is inflated by noise. Section 6 checks those with a model instead.

In [16]:
def income_table(col):
    """Share of each income level inside each group of `col` (rows sum to 1)."""
    t = pd.crosstab(df[col], df["income"], normalize="index")[INCOME_ORDER]
    if col == "age":
        t = t.reindex(AGE_ORDER)
    return t

CANDIDATES = ["age", "gender", "is_banned", "hour", "country", "requested_file"]

summary = pd.DataFrame({
    col: {
        "n_values": df[col].nunique(),
        "smallest_group": df[col].value_counts().min(),
        "max_deviation": (income_table(col) - income_dist).abs().max().max(),
    }
    for col in CANDIDATES
}).T
summary["max_deviation"] = summary["max_deviation"].astype(float).round(3)
summary

,n_values,smallest_group,max_deviation
age,8.0,12231.0,0.214
gender,2.0,49985.0,0.002
is_banned,2.0,4505.0,0.011
hour,24.0,4038.0,0.013
country,197.0,436.0,0.051
requested_file,11998.0,1.0,0.877


In [17]:
# age: each age group has only three income levels
income_table("age").round(2)

income,0-10k,10k-20k,20k-40k,40k-60k,60k-100k,100k-150k,150k-250k,250k+
age,,,,,,,,
0-16,0.34,0.33,0.00,0.00,0.00,0.00,0.00,0.33
17-25,0.34,0.33,0.33,0.00,0.00,0.00,0.00,0.00
26-35,0.00,0.33,0.33,0.34,0.00,0.00,0.00,0.00
36-45,0.00,0.00,0.34,0.33,0.33,0.00,0.00,0.00
46-55,0.00,0.00,0.00,0.33,0.33,0.34,0.00,0.00
56-65,0.00,0.00,0.00,0.00,0.34,0.33,0.33,0.00
66-75,0.00,0.00,0.00,0.00,0.00,0.34,0.33,0.34
76+,0.34,0.00,0.00,0.00,0.00,0.00,0.33,0.33


In [18]:
# gender and is_banned: every row looks like the overall distribution (about 0.125)
display(income_table("gender").round(2))
display(income_table("is_banned").round(2))
# hour has 24 rows and looks the same; uncomment to see it
# income_table("hour").round(2)

income,0-10k,10k-20k,20k-40k,40k-60k,60k-100k,100k-150k,150k-250k,250k+
gender,,,,,,,,
Female,0.13,0.12,0.12,0.13,0.13,0.13,0.12,0.12
Male,0.13,0.13,0.12,0.12,0.12,0.12,0.12,0.13


income,0-10k,10k-20k,20k-40k,40k-60k,60k-100k,100k-150k,150k-250k,250k+
is_banned,,,,,,,,
0,0.13,0.12,0.12,0.13,0.12,0.13,0.12,0.13
1,0.13,0.13,0.12,0.11,0.12,0.12,0.13,0.13


## 4. Combination: age × gender

A feature can be useless alone but useful together with another one.
If gender helped, some cells inside an age group would move away from 1/3.

In [19]:
age_gender = pd.crosstab([df["age"], df["gender"]], df["income"], normalize="index")[INCOME_ORDER]
age_gender = age_gender.reindex(AGE_ORDER, level=0)

# Only look at the non-zero cells: how far are they from 1/3?
nonzero = age_gender.values[age_gender.values > 0.05]
print(f"non-zero cells: min={nonzero.min():.3f}  max={nonzero.max():.3f}  (1/3 = 0.333)")
age_gender.round(2)

non-zero cells: min=0.321  max=0.348  (1/3 = 0.333)


income        0-10k  10k-20k  20k-40k  40k-60k  60k-100k  100k-150k  \
age   gender                                                          
0-16  Female   0.33     0.33     0.00     0.00      0.00       0.00   
      Male     0.34     0.33     0.00     0.00      0.00       0.00   
17-25 Female   0.35     0.33     0.32     0.00      0.00       0.00   
      Male     0.33     0.33     0.33     0.00      0.00       0.00   
26-35 Female   0.00     0.33     0.34     0.34      0.00       0.00   
      Male     0.00     0.33     0.32     0.34      0.00       0.00   
36-45 Female   0.00     0.00     0.33     0.33      0.34       0.00   
      Male     0.00     0.00     0.34     0.33      0.33       0.00   
46-55 Female   0.00     0.00     0.00     0.33      0.33       0.34   
      Male     0.00     0.00     0.00     0.34      0.33       0.34   
56-65 Female   0.00     0.00     0.00     0.00      0.33       0.34   
      Male     0.00     0.00     0.00     0.00      0.34       0.33   
66-75 Female   0.00     0.00     0.00     0.00      0.00       0.35   
      Male     0.00     0.00     0.00     0.00      0.00       0.33   
76+   Female   0.35     0.00     0.00     0.00      0.00       0.00   
      Male     0.33     0.00     0.00     0.00      0.00       0.00   

income        150k-250k  250k+  
age   gender                    
0-16  Female       0.00   0.34  
      Male         0.00   0.32  
17-25 Female       0.00   0.00  
      Male         0.00   0.00  
26-35 Female       0.00   0.00  
      Male         0.00   0.00  
36-45 Female       0.00   0.00  
      Male         0.00   0.00  
46-55 Female       0.00   0.00  
      Male         0.00   0.00  
56-65 Female       0.33   0.00  
      Male         0.33   0.00  
66-75 Female       0.32   0.33  
      Male         0.33   0.34  
76+   Female       0.32   0.33  
      Male         0.34   0.33

## 5. Ceiling when only age is used

The best possible rule with age is: for each age group, always guess its most common income.
Its accuracy on the full data is an upper bound for any model that uses only age.

In [20]:
counts = pd.crosstab(df["age"], df["income"]).reindex(AGE_ORDER)[INCOME_ORDER]
ceiling = counts.max(axis=1).sum() / counts.values.sum()

possible = counts.apply(
    lambda row: ", ".join(c for c in INCOME_ORDER if row[c] / row.sum() > 0.05), axis=1)

print(f"baseline (no features): {baseline:.4f}")
print(f"ceiling (age only):     {ceiling:.4f}")
possible.to_frame("possible incomes")

baseline (no features): 0.1274
ceiling (age only):     0.3381


,possible incomes
age,
0-16,"0-10k, 10k-20k, 250k+"
17-25,"0-10k, 10k-20k, 20k-40k"
26-35,"10k-20k, 20k-40k, 40k-60k"
36-45,"20k-40k, 40k-60k, 60k-100k"
46-55,"40k-60k, 60k-100k, 100k-150k"
56-65,"60k-100k, 100k-150k, 150k-250k"
66-75,"100k-150k, 150k-250k, 250k+"
76+,"0-10k, 150k-250k, 250k+"


## 6. Compare feature sets and models

Same split for every run: 80/20, stratified by income, one run per seed.
We record both **train** and **test** accuracy:
a large gap means the model memorizes noise (overfitting).

Random forest on all features is slow, so it uses fewer seeds.

In [21]:
def evaluate(features, make_model, seeds):
    """Train on each seed; return train/test accuracy per seed."""
    data = df[features + ["income"]].dropna()
    X = pd.get_dummies(data[features].astype(str))   # every feature treated as a category
    y = data["income"]
    rows = []
    for s in seeds:
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=0.2, random_state=s, stratify=y)
        m = make_model(s).fit(X_tr, y_tr)
        rows.append({"seed": s,
                     "train": accuracy_score(y_tr, m.predict(X_tr)),
                     "test": accuracy_score(y_te, m.predict(X_te))})
    return pd.DataFrame(rows)

FEATURE_SETS = {
    "all features": ["gender", "age", "country", "is_banned", "hour"],
    "age only":     ["age"],
    "age + gender": ["age", "gender"],
    "age + country": ["age", "country"],
    "age + hour":   ["age", "hour"],
}
MODELS = {
    "decision tree": (lambda s: DecisionTreeClassifier(random_state=s), SEEDS),
    "random forest": (lambda s: RandomForestClassifier(n_estimators=100, n_jobs=-1,
                                                       random_state=s), SEEDS[:5]),
}

results = []
for model_name, (make_model, seeds) in MODELS.items():
    for set_name, feats in FEATURE_SETS.items():
        t0 = time.time()
        r = evaluate(feats, make_model, seeds)
        results.append({
            "model": model_name, "features": set_name, "seeds": len(seeds),
            "train_mean": r["train"].mean(),
            "test_mean": r["test"].mean(), "test_std": r["test"].std(),
            "test_min": r["test"].min(), "test_max": r["test"].max(),
            f"runs >= {TARGET_ACC:.0%}": int((r["test"] >= TARGET_ACC).sum()),
        })
        print(f"{model_name:14s} {set_name:14s} done in {time.time() - t0:.1f}s")

comparison = pd.DataFrame(results).round(4)
comparison

decision tree  all features   done in 35.4s
decision tree  age only       done in 2.0s
decision tree  age + gender   done in 2.0s
decision tree  age + country  done in 30.4s
decision tree  age + hour     done in 2.6s
random forest  all features   done in 18.8s
random forest  age only       done in 2.0s
random forest  age + gender   done in 2.1s
random forest  age + country  done in 14.8s
random forest  age + hour     done in 2.7s


,model,features,seeds,train_mean,test_mean,test_std,test_min,test_max,runs >= 35%
0,decision tree,all features,10,0.7683,0.3353,0.0031,0.3280,0.3387,0
1,decision tree,age only,10,0.3383,0.3362,0.0020,0.3334,0.3388,0
2,decision tree,age + gender,10,0.3406,0.3362,0.0039,0.3316,0.3437,0
3,decision tree,age + country,10,0.4020,0.3340,0.0026,0.3270,0.3360,0
4,decision tree,age + hour,10,0.3572,0.3345,0.0028,0.3311,0.3400,0
5,random forest,all features,5,0.7683,0.3358,0.0011,0.3342,0.3368,0
6,random forest,age only,5,0.3386,0.3355,0.0021,0.3334,0.3388,0
7,random forest,age + gender,5,0.3408,0.3361,0.0025,0.3324,0.3390,0
8,random forest,age + country,5,0.4014,0.3347,0.0018,0.3328,0.3376,0
9,random forest,age + hour,5,0.3568,0.3358,0.0019,0.3335,0.3372,0


## 7. Check the final code

Run the function that `main.py` actually uses, so the numbers in the report come from the real pipeline.
Update `FEATURES` in `src/income_model.py` first if you changed your choice above.

In [22]:
from income_model import model as train_income_model, FEATURES

final = pd.DataFrame(
    [{"seed": s, "test": train_income_model(df, seed=s)[0]} for s in SEEDS])

print(f"FEATURES in src/income_model.py: {FEATURES}")
print(f"test accuracy: mean={final['test'].mean():.4f}  "
      f"range=[{final['test'].min():.4f}, {final['test'].max():.4f}]  "
      f"runs >= {TARGET_ACC:.0%}: {(final['test'] >= TARGET_ACC).sum()}/{len(final)}")
final.round(4)

FEATURES in src/income_model.py: ['gender', 'age', 'country', 'is_banned', 'hour']
test accuracy: mean=0.3353  range=[0.3311, 0.3384]  runs >= 35%: 0/10


,seed,test
0,0,0.3369
1,1,0.3361
2,2,0.3347
3,3,0.3360
4,4,0.3345
5,5,0.3369
6,6,0.3358
7,7,0.3312
8,8,0.3330
9,9,0.3384


## 8. Numbers for the report

Copy these rows into the LaTeX tables.

In [23]:
print(f"Rows: {len(df)}   Baseline: {baseline:.4f}   Age-only ceiling: {ceiling:.4f}\n")

print("% Table: possible incomes per age group")
for age, incomes in possible.items():
    print(f"{age} & {incomes} \\\\")

print("\n% Table: model comparison (test accuracy over seeds)")
for _, r in comparison.iterrows():
    print(f"{r['model']} & {r['features']} & {r['train_mean']:.4f} & "
          f"{r['test_mean']:.4f} & [{r['test_min']:.4f}, {r['test_max']:.4f}] \\\\")

Rows: 99999   Baseline: 0.1274   Age-only ceiling: 0.3381

% Table: possible incomes per age group
0-16 & 0-10k, 10k-20k, 250k+ \\
17-25 & 0-10k, 10k-20k, 20k-40k \\
26-35 & 10k-20k, 20k-40k, 40k-60k \\
36-45 & 20k-40k, 40k-60k, 60k-100k \\
46-55 & 40k-60k, 60k-100k, 100k-150k \\
56-65 & 60k-100k, 100k-150k, 150k-250k \\
66-75 & 100k-150k, 150k-250k, 250k+ \\
76+ & 0-10k, 150k-250k, 250k+ \\

% Table: model comparison (test accuracy over seeds)
decision tree & all features & 0.7683 & 0.3353 & [0.3280, 0.3387] \\
decision tree & age only & 0.3383 & 0.3362 & [0.3334, 0.3388] \\
decision tree & age + gender & 0.3406 & 0.3362 & [0.3316, 0.3437] \\
decision tree & age + country & 0.4020 & 0.3340 & [0.3270, 0.3360] \\
decision tree & age + hour & 0.3572 & 0.3345 & [0.3311, 0.3400] \\
random forest & all features & 0.7683 & 0.3358 & [0.3342, 0.3368] \\
random forest & age only & 0.3386 & 0.3355 & [0.3334, 0.3388] \\
random forest & age + gender & 0.3408 & 0.3361 & [0.3324, 0.3390] \\
random f